**PROJET 2 : MACHINE LEARNING**

**APPRENTISSAGE HYBRIDE POUR LA CLASSIFICATION DES MALADIES DES FEUILLES DE CAFÉ À PARTIR DE DONNÉES MULTISPECTRALES ET INDICES DE VÉGÉTATION**

**Nom et Prénom : Traoré Ismael**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**0-INSPECTION & PRÉPARATION INITIALE**

In [ ]:
# 0. INSPECTION & PRÉPARATION INITIALE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 0.1 Chargement du dataset

file_path = "data_Projet2.xlsx"

df = pd.read_excel(file_path)

# 0.2 Inspection générale

print("DIMENSIONS DU DATASET ")
print(df.shape)

print("\nPREMIÈRES LIGNES ")
display(df.head())

print("\nINFORMATIONS GÉNÉRALES ")
df.info()

# 0.3 Qualité des données

print("\n=== VALEURS MANQUANTES ===")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\n=== DOUBLONS ===")
print(df.duplicated().sum())

# 0.4 Séparation X / y

X = df.drop(columns=['Class'])
y = df['Class']

print("\n DIMENSIONS X / y ")
print(X.shape, y.shape)

# 0.5 Analyse rapide de la cible (IMPORTANT pour EDA)

print("\n=== DISTRIBUTION DES CLASSES ===")
print(y.value_counts())

print("\n=== POURCENTAGES ===")
print(y.value_counts(normalize=True) * 100)

plt.figure(figsize=(6,4))
sns.countplot(x=y)
plt.title("Distribution des classes (Class)")
plt.xticks(rotation=45)
plt.show()

On observe 51 valeurs manquantes dans EVI_mean et EVI_std.
La plupart des algorithmes (PCA, t-SNE, K-Means, GMM) ne fonctionnent pas avec des valeurs manquantes.

Ce que nous faisons

Comme il s’agit d’un indice de végétation, nous avons deux options :

- supprimer ces lignes (≈ 4,5 % du dataset)
- ou imputer les valeurs.

Nous choisissons l’imputation par la médiane, car elle est plus stable et conserve mieux les données.


Déséquilibre des classes
- rust : 75.65 %
- norust : 24.35 %

Le dataset est déséquilibré environ 3:1. Cela peut biaiser les modèles vers la classe majoritaire.

On utilisera plus tard des métriques adaptées :

- F1-score macro
- Balanced Accuracy
- MCC
- alidation stratifiée

In [ ]:
0.6 # Traitement des valeurs manquantes par imputation (Médiane)
# Nous nettoyons les données en imputant les valeurs manquantes avec la médiane.
# Nous remplaçons les valeurs manquantes de chaque colonne par sa médiane
X = X.fillna(X.median())

print("=== VÉRIFICATION APRÈS IMPUTATION ===")
print(f"Nombre de valeurs manquantes restantes dans X : {X.isnull().sum().sum()}")

Les valeurs manquantes ont été correctement imputées.
Le dataset est désormais complet et exploitable pour l’ensemble des étapes du pipeline réduction de dimension, clustering et classification.

**I - ANALYSE EXPLORATOIRE (EDA)**

In [ ]:
# I — 1. STATISTIQUES DESCRIPTIVES & DISTRIBUTIONS

# 1.1 Calcul des statistiques descriptives clés
print(" STATISTIQUES DESCRIPTIVES DES FEATURES ")
stats = X.describe().T[['mean', 'std', 'min', 'max']]
display(stats.round(3))

# 1.2 Liste des moyennes des bandes spectrales principales
bandes_mean = ['Blue_mean', 'Green_mean', 'Red_mean', 'RedEdge_mean', 'NIR_mean']

In [ ]:
# 1.3 Histogrammes des distributions spectrales
plt.figure(figsize=(12, 8))
for i, col in enumerate(bandes_mean, 1):
    plt.subplot(2, 3, i)
    sns.histplot(X[col], kde=True, color='green')
    plt.title(f"Distribution : {col}")
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Boxplots pour visualiser les échelles et les outliers
plt.figure(figsize=(10, 5))
sns.boxplot(data=X[bandes_mean], palette="Set2")
plt.title("Boxplots des bandes spectrales (Moyennes)")
plt.grid(alpha=0.3)
plt.show()

Nous observons que les variables présentent des échelles différentes, notamment entre les bandes spectrales et les indices de végétation. Cela rend la standardisation nécessaire avant toute méthode basée sur les distances comme le clustering ou PCA.

Les distributions montrent également une certaine asymétrie et la présence de valeurs extrêmes. Ces variations peuvent refléter la diversité des états des feuilles ainsi que des conditions de capture des données. Une analyse plus poussée sera nécessaire pour relier ces variations aux classes de maladies.

I — 1.5 ANALYSE DES CORRÉLATIONS PEARSON & SPEARMAN

In [ ]:
# I — 1.5 ANALYSE DES CORRÉLATIONS (PEARSON & SPEARMAN)

# 1.5.1 Calcul des matrices de corrélation
corr_pearson = X.corr(method='pearson')
corr_spearman = X.corr(method='spearman')

# 1.5.2 Heatmap pour la corrélation de Pearson
plt.figure(figsize=(12, 10))
sns.heatmap(corr_pearson, cmap='coolwarm', vmin=-1, vmax=1, center=0, annot=False)
plt.title("Matrice de corrélation de Pearson (Relations linéaires)")
plt.show()


In [ ]:
# 1.5.3 Heatmap pour la corrélation de Spearman
plt.figure(figsize=(12, 10))
sns.heatmap(corr_spearman, cmap='coolwarm', vmin=-1, vmax=1, center=0, annot=False)
plt.title("Matrice de corrélation de Spearman (Relations monotones)")
plt.show()

Ce qu’on observe :

Forte colinéarité : Plusieurs variables présentent des corrélations élevées positives ou négatives, indiquant une redondance importante de l’information entre bandes spectrales et indices de végétation.

Pearson vs Spearman : Les deux matrices sont très similaires, ce qui suggère que les relations entre variables sont principalement stables, à la fois linéaires et monotones.

Cela confirme la nécessité d’une réduction de dimension (PCA) avant les étapes de clustering et de classification.

**1-6 La Multicolinéarité (VIF - Variance Inflation Factor)**

Pour mesurer mathématiquement cette forte redondance constatée visuellement

In [ ]:

#  1.6 MULTICOLINÉARITÉ (VIF)

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Nous créons le DataFrame pour stocker les scores VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns

# Nous calculons le VIF directement sur les valeurs de X
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# Nous affichons les variables triées par le score VIF décroissant
print("SCORES VIF DES VARIABLES (MÉTHODE PROPRE) ")
display(vif_data.sort_values(by="VIF", ascending=False))

Les résultats du VIF révèlent une multicolinéarité très forte entre plusieurs indices de végétation :

Colinéarité extrême (VIF > 10 000) : Les variables NDVI_mean / SAVI_mean ainsi que NDVI_std / SAVI_std présentent des valeurs de VIF très élevées. Cela s’explique par le fait que ces indices sont fortement liés mathématiquement et dérivés des mêmes bandes spectrales.

Forte redondance générale (VIF > 10) : Une grande partie des bandes spectrales et des variables dérivées présentent des VIF élevés, indiquant une forte dépendance entre les features et une redondance de l’information.

Variables plus stables (VIF < 10) : Certaines variables comme PSRI, NDRE et EVI montrent des valeurs de VIF plus faibles, suggérant une contribution plus spécifique et moins redondante.

I - 2. ANALYSE DE STRUCTURE (RÉDUCTION DE DIMENSION)

Comme nous l'avons constaté avec le VIF, nos variables ont des échelles très différentes (les bandes spectrales atteignent des milliers, tandis que les indices varient entre -1 et 1). Avant d'appliquer la PCA, l'UMAP ou le t-SNE, nous devons obligatoirement standardiser les données (centrer-réduire) pour qu'aucune variable n'écrase artificiellement les autres.

In [ ]:
# I — 2. ANALYSE DE STRUCTURE (RÉDUCTION DE DIMENSION)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

# 2.1 Nous appliquons la standardisation pour mettre les variables à la même échelle
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# 2.2 Nous calculons la PCA pour capturer la structure globale linéaire
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)


In [ ]:
# 2.3 Nous calculons le t-SNE avec une initialisation PCA pour la stabilité locale
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_scaled)

In [ ]:
# 2.4 Nous calculons l'UMAP pour préserver à la fois le global et le local
umap_model = UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

In [ ]:
# 2.5 Visualisation comparative des trois projections
plt.figure(figsize=(18, 5))

# Affichage PCA
plt.subplot(1, 3, 1)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, alpha=0.7)
plt.title("PCA")

# Affichage t-SNE
plt.subplot(1, 3, 2)
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y, alpha=0.7)
plt.title("t-SNE")

# Affichage UMAP
plt.subplot(1, 3, 3)
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=y, alpha=0.7)
plt.title("UMAP")

plt.tight_layout()
plt.show()

Nous constatons que Les projections PCA, t-SNE et UMAP montrent un chevauchement important entre les classes rust et norust, avec des zones de mélange visibles.

D'ou Les classes ne sont pas parfaitement séparables dans l’espace des features. Cela indique une structure complexe des données et justifie l’utilisation d’approches hybrides combinant clustering, consensus et apprentissage supervisé.

**II- APPRENTISSAGE NON SUPERVISÉ MULTI-MODÈLES**

II.1 : Optimisation et application du K-Means

Avant de tester l’ensemble des modèles, nous déterminons le nombre optimal de clusters (k) pour K-Means. Ce choix servira également de référence pour les autres algorithmes de clustering.

Bien que la variable cible comporte 2 classes (rust et norust), nous testons plusieurs valeurs de k (de 2 à 6) afin d’analyser la structure interne des données de manière exploratoire.

In [ ]:
# PARTIE II — APPRENTISSAGE NON SUPERVISÉ MULTI-MODÈLES

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Nous préparons des listes pour stocker nos résultats d'optimisation
k_range = range(2, 7)
scores_silhouette = []
scores_db = []
scores_ch = []

print(" OPTIMISATION DU NOMBRE DE CLUSTERS (K-MEANS)")

# Nous testons différentes configurations de K-Means de K=2 à K=6
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)

    # Nous calculons toutes les métriques imposées par le sujet
    sil = silhouette_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    ch = calinski_harabasz_score(X_scaled, labels)

    scores_silhouette.append(sil)
    scores_db.append(db)
    scores_ch.append(ch)

    print(f"Pour K = {k} -> Silhouette : {sil:.4f} | Davies-Bouldin : {db:.4f} | Calinski-Harabasz : {ch:.2f}")

In [ ]:
# Visualisation des courbes de décision pour le choix de K
plt.figure(figsize=(15, 4))

# Courbe Silhouette
plt.subplot(1, 3, 1)
plt.plot(k_range, scores_silhouette, marker='o', color='b')
plt.title("Silhouette (Plus haut est le mieux)")
plt.xlabel("Nombre de clusters (K)")
plt.grid(alpha=0.3)

# Courbe Davies-Bouldin
plt.subplot(1, 3, 2)
plt.plot(k_range, scores_db, marker='o', color='r')
plt.title("Index Davies-Bouldin (Plus bas est le mieux)")
plt.xlabel("Nombre de clusters (K)")
plt.grid(alpha=0.3)

# Courbe Calinski-Harabasz
plt.subplot(1, 3, 3)
plt.plot(k_range, scores_ch, marker='o', color='g')
plt.title("Score Calinski-Harabasz (Plus haut est le mieux)")
plt.xlabel("Nombre de clusters (K)")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Analyse l’optimisation de K :

- Candidat statistique (K = 3) : Ce choix présente les meilleures performances en clustering pur, avec un score de silhouette maximal (0.6517) et un indice de Davies-Bouldin minimal (1.2123), indiquant une bonne séparation des groupes.

- Candidat interprétable (K = 2) : Les performances restent très proches, avec une silhouette élevée (0.6487) et un bon score de Calinski-Harabasz (416.25), suggérant une structure également cohérente.

Conclusion : Bien que K = 3 optimise légèrement les métriques internes, nous retenons K = 2 pour la suite afin d’assurer une cohérence avec la variable cible (rust / norust) et faciliter la comparaison entre clustering non supervisé et labels réels.

**II —2 ENTRAÎNEMENT DES 5 MODÈLES EN PARALLÈLE & TABLEAU COMPARATIF DES PERFORMANCES**

Maintenant que $K=2$ est validé, nous allons faire tourner les 5 algorithmes demandés

In [ ]:
# MULTI-MODÈLES & TABLEAU COMPARATIF DES PERFORMANCES

from sklearn.cluster import KMeans, SpectralClustering, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)

# 1. Nous initialisons les 5 algorithmes d'apprentissage non supervisé requis
modeles = {
    "K-Means": KMeans(n_clusters=2, random_state=42, n_init=10, max_iter=300),
    "GMM": GaussianMixture(n_components=2, random_state=42),
    "Spectral": SpectralClustering(n_clusters=2, random_state=42, assign_labels='kmeans'),
    "DBSCAN": DBSCAN(eps=3.5, min_samples=5),
    "Agglomerative": AgglomerativeClustering(n_clusters=2)
}

In [ ]:
# 2. Dictionnaire pour centraliser les performances
dict_perf = {}

print(" ENTRAÎNEMENT ET ÉVALUATION DES 5 MODÈLES ")

In [ ]:
# 3. Boucle d'entraînement unique et calcul des métriques
for nom, model in modeles.items():
    labels = model.fit_predict(X_scaled)

    # Nous comptons le nombre de clusters valides trouvés (en ignorant le bruit -1)
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

    # Nous calculons les scores uniquement si le modèle sépare au moins 2 groupes
    if n_clusters_ > 1:
        sil = silhouette_score(X_scaled, labels)
        db = davies_bouldin_score(X_scaled, labels)
        ch = calinski_harabasz_score(X_scaled, labels)
        ari = adjusted_rand_score(y, labels)
        nmi = normalized_mutual_info_score(y, labels)
    else:
        sil, db, ch, ari, nmi = None, None, None, None, None
        print(f"⚠️ {nom} : Un seul cluster détecté (ou trop de bruit). Ajuster les paramètres.")

    # Stockage des indicateurs
    dict_perf[nom] = {
        "Silhouette": sil,
        "Davies-Bouldin": db,
        "Calinski-Harabasz": ch,
        "ARI": ari,
        "NMI": nmi
    }

**TABLEAU COMPARATIF DES PERFORMANCES**

In [ ]:
# 4. Génération et affichage du tableau de synthèse
df_performances = pd.DataFrame(dict_perf).T
display(df_performances.round(4))



 Analyse des performances des modèles :
- Vainqueur Métier (GMM & Agglomerative) : Le GMM (suivi de très près par l'Agglomerative) écrase les autres sur la correspondance avec le terrain avec un ARI de 0.7046 et un NMI de 0.6186. C'est le modèle qui capture le mieux la distinction biologique entre les feuilles saines et malades.

- Le Piège Géométrique (Spectral Clustering) : Il obtient une Silhouette impressionnante de 0.8328, mais ses scores ARI et NMI sont proches de 0. Cela signifie qu'il a créé un groupe géométriquement parfait en apparence, mais qui ne correspond absolument pas à la réalité des classes médicales.

- Le compromis standard (K-Means) : Bon comportement géométrique global, mais moins précis que le GMM pour isoler la rouille.

Le modèle GMM est sélectionné comme notre modèle de référence (baseline non supervisée). Sa supériorité s'explique par sa capacité à gérer des clusters de formes elliptiques et à modéliser des frontières probabilistes, ce qui correspond parfaitement au chevauchement spectral des signatures de feuilles malades.

II — 3. CONSENSUS CLUSTERING & VOTE MAJORITAIRE

In [ ]:
# II — 3. CONSENSUS CLUSTERING & VOTE MAJORITAIRE (CORRIGÉ SANS SCIPY)

import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# 3.1 Nous réutilisons les prédictions de nos 3 meilleurs modèles
gmm_labels = GaussianMixture(n_components=2, random_state=42).fit_predict(X_scaled)
agg_labels = AgglomerativeClustering(n_clusters=2).fit_predict(X_scaled)
kmeans_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_scaled)

# 3.2 Fonction de mapping utilisant np.unique (compatible chaînes de caractères/texte)
def aligner_labels(cluster_labels, true_labels):
    aligned = np.zeros_like(cluster_labels, dtype=object) # dtype=object pour accepter le texte
    true_labels_arr = np.array(true_labels)

    for cluster in np.unique(cluster_labels):
        mask = (cluster_labels == cluster)
        # Extraction du mode/majoritaire de manière native avec NumPy
        valeurs, comptes = np.unique(true_labels_arr[mask], return_counts=True)
        true_mode = valeurs[np.argmax(comptes)]
        aligned[mask] = true_mode
    return aligned

# Alignement post-hoc des prédictions textuelles
gmm_aligned = aligner_labels(gmm_labels, y)
agg_aligned = aligner_labels(agg_labels, y)
kmeans_aligned = aligner_labels(kmeans_labels, y)

# 3.3 Vote majoritaire pour obtenir le consensus par ligne
stacked_predictions = np.column_stack((gmm_aligned, agg_aligned, kmeans_aligned))
consensus_labels = []

for row in stacked_predictions:
    valeurs, comptes = np.unique(row, return_counts=True)
    consensus_labels.append(valeurs[np.argmax(comptes)])

consensus_labels = np.array(consensus_labels)

# 3.4 Calcul des performances finales du Consensus
ari_consensus = adjusted_rand_score(y, consensus_labels)
nmi_consensus = normalized_mutual_info_score(y, consensus_labels)

print(" PERFORMANCES DU CONSENSUS CLUSTERING ")
print(f"ARI du Consensus : {ari_consensus:.4f}")
print(f"NMI du Consensus : {nmi_consensus:.4f}")

Analyse du Consensus Clustering :

- ARI de 0.6900 et NMI de 0.6048 : Le consensus se stabilise au niveau des meilleures performances individuelles (quasiment égal au GMM qui avait le score maximum de 0.7046).

- Lissage du bruit : En combinant les décisions, le vote majoritaire permet d'éliminer les prédictions aberrantes ou isolées des modèles pris séparément. On obtient ainsi des pseudo-étiquettes extrêmement robustes et ancrées dans la réalité physique de nos données.

**III- FEATURE ENGINEERING BASÉ SUR LE CLUSTERING**\

In [ ]:
# PARTIE IV — FEATURE ENGINEERING BASÉ SUR LE CLUSTERING (100% CONFORME)

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# 3.1 Encodage numérique de la feature de consensus (0 ou 1)
consensus_numeric = np.where(consensus_labels == 'rust', 1, 0)

# 3.2 Probabilités d'appartenance aux clusters du GMM
gmm_model = GaussianMixture(n_components=2, random_state=42)
gmm_model.fit(X_scaled)
gmm_probabilities = gmm_model.predict_proba(X_scaled)

# 3.3 Distances euclidiennes aux centroïdes du K-Means
kmeans_model = KMeans(n_clusters=2, random_state=42, n_init=10, max_iter=300)
kmeans_model.fit(X_scaled)
kmeans_distances = kmeans_model.transform(X_scaled)

# 3.4 Création et enrichissement de la matrice finale
X_enriched = X.copy()

# Ajout des étiquettes (Cluster Labels)
X_enriched['cluster_consensus'] = consensus_numeric

# Ajout des probabilités GMM
X_enriched['gmm_proba_cl0'] = gmm_probabilities[:, 0]
X_enriched['gmm_proba_cl1'] = gmm_probabilities[:, 1]

# Ajout des distances aux centroïdes K-Means
X_enriched['dist_cluster_0'] = kmeans_distances[:, 0]
X_enriched['dist_cluster_1'] = kmeans_distances[:, 1]

# Ajout des embeddings PCA (déjà calculés en Partie I)
X_enriched['PCA_1'] = X_pca[:, 0]
X_enriched['PCA_2'] = X_pca[:, 1]

# Ajout des embeddings UMAP (déjà calculés en Partie I)
X_enriched['UMAP_1'] = X_umap[:, 0]
X_enriched['UMAP_2'] = X_umap[:, 1]

# 3.5 Vérification finale des dimensions et de la structure
print(" VÉRIFICATION DU DATASET ENRICHI (VERSION COMPLÈTE)")
print(f"Dimensions initiales de X : {X.shape}")
print(f"Dimensions de X_enriched  : {X_enriched.shape}")
print(f"Nombre total de nouvelles variables créées : {X_enriched.shape[1] - X.shape[1]}")

print("\nAperçu des nouvelles variables ajoutées au dataset :")
colonnes_crees = ['cluster_consensus', 'gmm_proba_cl0', 'gmm_proba_cl1',
                  'dist_cluster_0', 'dist_cluster_1', 'PCA_1', 'PCA_2', 'UMAP_1', 'UMAP_2']
display(X_enriched[colonnes_crees].head())

Les dimensions du dataset enrichi sont conformes (1121, 39). Les probabilités du GMM, les distances aux centroïdes de K-Means ainsi que les embeddings PCA et UMAP ont été correctement générés. Le jeu de données enrichi est désormais prêt pour la phase de classification supervisée hybride.

**# PARTIE IV - CLASSIFICATION SUPERVISÉE HYBRIDE**

In [ ]:
# CLASSIFICATION SUPERVISÉE HYBRIDE

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 4.1 Encodage numérique global de la cible (0 = norust, 1 = rust)
y_numeric = np.where(y == 'rust', 1, 0)

# 4.2 Initialisation des 5 classifieurs imposés
classifieurs = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1),
    "SVM RBF": SVC(kernel='rbf', probability=True, random_state=42),
    "MLP (Réseau de neurones)": MLPClassifier(max_iter=500, random_state=42)
}

# 4.3 Configuration de la validation croisée stratifiée à 5 plis et métriques pondérées
cv_strategie = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
les_metriques = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'roc_auc']

# Structure pour stocker les résultats des deux scénarios
resultats_scenarios = {}

# 4.4 Boucle d'évaluation comparative (Original vs Enrichi)
for scenario, X_data in {"Original": X, "Enrichi": X_enriched}.items():
    print(f"\n=== ÉVALUATION DU SCÉNARIO : {scenario} ===")
    resultats_scenarios[scenario] = {}

    for nom, clf in classifieurs.items():
        scores = cross_validate(clf, X_data, y_numeric, cv=cv_strategie, scoring=les_metriques)

        # Stockage de la Performance Moyenne (± Écart-Type)
        resultats_scenarios[scenario][nom] = {
            "Accuracy": f"{np.mean(scores['test_accuracy']):.4f} (±{np.std(scores['test_accuracy']):.3f})",
            "F1-Score": f"{np.mean(scores['test_f1_weighted']):.4f} (±{np.std(scores['test_f1_weighted']):.3f})",
            "AUC-ROC": f"{np.mean(scores['test_roc_auc']):.4f} (±{np.std(scores['test_roc_auc']):.3f})"
        }
        print(f"✓ {nom} calculé.")

# 4.5 Affichage des tableaux de synthèse
for scenario, data in resultats_scenarios.items():
    print(f"\ TABLEAU DE SYNTHÈSE - SCÉNARIO : {scenario}")
    df_res = pd.DataFrame(data).T
    display(df_res)

Les modèles basés sur les arbres de décision (Random Forest, XGBoost et LightGBM) présentent les meilleures performances globales, avec des scores d’Accuracy et de F1-Score supérieurs à 91 %.

L’apport du feature engineering basé sur le clustering est particulièrement visible sur XGBoost et LightGBM, avec une amélioration des performances ainsi qu’une légère réduction de la variance, ce qui indique une meilleure stabilité du modèle.

Le SVM RBF reste globalement inchangé, montrant une faible sensibilité aux nouvelles variables ajoutées.

Le MLP affiche des performances faibles, probablement en raison de la taille limitée du dataset, de la forte colinéarité des variables et de l’absence d’optimisation des hyperparamètres.

Conclusion : les meilleurs modèles pour ce problème sont Random Forest et LightGBM sur le scénario enrichi, offrant le meilleur compromis entre performance et stabilité.

In [ ]:

# V — PSEUDO-LABELING & CO-LEARNING (VERSION ACADÉMIQUE SANS BIAIS)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 5.1 Séparation stricte Train / Test pour éviter le biais d'évaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_enriched, y_numeric, test_size=0.3, random_state=42, stratify=y_numeric
)

# 5.2 Entraînement initial du classifieur de référence sur le sous-ensemble Train
base_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
base_clf.fit(X_train, y_train)

# Évaluation initiale sur le sous-ensemble Test (la référence neutre)
preds_test_initial = base_clf.predict(X_test)
score_initial = accuracy_score(y_test, preds_test_initial)

# 5.3 Génération des probabilités et prédictions sur l'ensemble Test
probas_test = base_clf.predict_proba(X_test)
preds_test = base_clf.predict(X_test)
confiance_max = np.max(probas_test, axis=1)

# Application du seuil strict de haute confiance à 95%
seuil_confiance = 0.95
masque_fiable = confiance_max >= seuil_confiance
pct_fiable = np.mean(masque_fiable) * 100

print("=== ANALYSE DE LA CONFIANCE DES PRÉDICTIONS ===")
print(f"Pourcentage de données Test à haute confiance (>= {seuil_confiance*100}%) : {pct_fiable:.2f}%")

# 5.4 Création du set enrichi par Pseudo-Labeling (Données Train + Pseudo-labels du Test)
X_test_fiable = X_test[masque_fiable]
y_test_pseudo = preds_test[masque_fiable]

# Fusion des données d'origine avec les nouveaux pseudo-labels
X_train_augmented = pd.concat([X_train, X_test_fiable])
y_train_augmented = np.concatenate([y_train, y_test_pseudo])

# 5.5 Réentraînement du modèle (Co-learning) sur le dataset augmenté
pseudo_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
pseudo_clf.fit(X_train_augmented, y_train_augmented)

# Évaluation finale sur l'intégralité du Test original pour mesurer le vrai gain
preds_test_final = pseudo_clf.predict(X_test)
score_final = accuracy_score(y_test, preds_test_final)

print("\n=== ÉVALUATION DU GAIN / PERTE DE PERFORMANCE DE L'APPROCHE HYBRIDE ===")
print(f"Accuracy du modèle initial (Train pur)           : {score_initial:.4f}")
print(f"Accuracy du modèle augmenté (Pseudo-Labeling)     : {score_final:.4f}")
print(f"Différence nette de performance                  : {score_final - score_initial:+.4f}")

 Le modèle n'a sélectionné que la moitié des données du jeu de test (celles où il était sûr à plus de 95 %). Ce seuil strict est une bonne sécurité pour éviter d'injecter des erreurs dans le modèle.

 L'Accuracy reste exactement la même (0.9080) avant et après le pseudo-labeling.

Les 51 % de données réinjectées étaient déjà les exemples les plus "faciles" et les plus évidents. Le modèle les connaissait déjà, donc cela ne lui a rien apporté de nouveau pour corriger ses erreurs sur les 49 % de données difficiles restantes.

La performance n'a pas baissé. Le filtre a bien joué son rôle en empêchant le modèle de s'auto-tromper en propageant de fausses étiquettes.

**VII - ÉVALUATION FINALE & VALIDATION STATISTIQUE**

In [ ]:
# PARTIE VI — ÉVALUATION FINALE & VALIDATION STATISTIQUE (AVEC MLP)


import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# 6.1 Définition de toutes les métriques demandées
scores_avances = {
    'accuracy': make_scorer(accuracy_score),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'precision_weighted': make_scorer(precision_score, average='weighted', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro'),
    'recall_weighted': make_scorer(recall_score, average='weighted'),
    'f1_macro': make_scorer(f1_score, average='macro'),
    'f1_weighted': make_scorer(f1_score, average='weighted'),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'mcc': make_scorer(matthews_corrcoef),
    'cohen_kappa': make_scorer(cohen_kappa_score)
}

# 6.2 Validation croisée hautement robuste (5 plis x 3 répétitions = 15 évaluations)
cv_robuste = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Liste des 5 modèles du projet pour les tests statistiques
les_modeles = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1),
    "SVM RBF": SVC(kernel='rbf', random_state=42),
    "MLP (Réseau de neurones)": MLPClassifier(max_iter=500, random_state=42)
}

dict_final = {}
scores_bruts = {}

print(" CALCUL DES MÉTRIQUES AVANCÉES SUR 15 FOLDS ")

for nom, clf in les_modeles.items():
    scores = cross_validate(clf, X_enriched, y_numeric, cv=cv_robuste, scoring=scores_avances)
    scores_bruts[nom] = scores['test_accuracy']  # Vecteur de 15 scores pour la statistique

    dict_final[nom] = {
        "Accuracy": np.mean(scores['test_accuracy']),
        "Precision (Macro)": np.mean(scores['test_precision_macro']),
        "Precision (Weighted)": np.mean(scores['test_precision_weighted']),
        "Recall (Macro)": np.mean(scores['test_recall_macro']),
        "Recall (Weighted)": np.mean(scores['test_recall_weighted']),
        "F1-Score (Macro)": np.mean(scores['test_f1_macro']),
        "F1-Score (Weighted)": np.mean(scores['test_f1_weighted']),
        "Balanced Accuracy": np.mean(scores['test_balanced_accuracy']),
        "MCC": np.mean(scores['test_mcc']),
        "Cohen Kappa": np.mean(scores['test_cohen_kappa'])
    }
    print(f"✓ {nom} évalué.")

# Affichage du tableau de synthèse complet
df_final = pd.DataFrame(dict_final).T
print("\n TABLEAU COMPARATIF FINAL DES MÉTRIQUES :")
display(df_final.round(4))

# 6.3 Tests Statistiques de Validation
print("\n TESTS STATISTIQUES DE VALIDATION ")

# 1. Test de Friedman (Comparaison globale de TOUS les modèles)
stat_f, p_val_f = friedmanchisquare(
    scores_bruts["Random Forest"],
    scores_bruts["XGBoost"],
    scores_bruts["LightGBM"],
    scores_bruts["SVM RBF"],
    scores_bruts["MLP (Réseau de neurones)"]
)
print(f"Test global de Friedman : p-value = {p_val_f:.5f}")
if p_val_f < 0.05:
    print(" Significatif (p < 0.05) : Au moins un modèle se démarque significativement des autres.")
else:
    print("Non significatif (p >= 0.05) : Les performances globales des modèles sont équivalentes.")

# 2. Test de Wilcoxon (Comparaison paire par paire sur les deux meilleurs modeles)
stat_w, p_val_w = wilcoxon(scores_bruts["Random Forest"], scores_bruts["LightGBM"])
print(f"\nTest de Wilcoxon (RF vs LightGBM) : p-value = {p_val_w:.5f}")
if p_val_w < 0.05:
    print(" Différence statistiquement significative entre Random Forest et LightGBM.")
else:
    print("Pas de différence statistiquement significative (les deux leaders se valent).")

- Les meilleurs modèles : Random Forest et LightGBM ont les meilleurs résultats avec 92.92 % d'Accuracy. Les scores MCC (0.80) et Kappa (0.78) prouvent qu'ils gèrent très bien le déséquilibre des classes de notre jeu de données.

- XGBoost : Il se place juste derrière avec 92.36 % d'Accuracy. C'est aussi une très bonne option.

- Le flop (SVM et MLP) : Le SVM ne détecte rien de stable (Kappa à 0) et le MLP reste très faible avec seulement 50.43 % d'Accuracy. Ils ne profitent pas du tout des nouvelles variables du clustering.

**les tests statistiques :**

- Test de Friedman (p-value = 0.00000) : La différence globale est significative. Cela montre scientifiquement que tous les modèles ne se valent pas. Les modèles d'arbres sont nettement supérieurs aux autres.

- Test de Wilcoxon (p-value = 0.97192) : La différence entre Random Forest et LightGBM n'est pas significative. D'un point de vue  statistique, les deux modèles se valent. On peut choisir l'un ou l'autre pour notre pipeline final.


**VII - INTERPRÉTABILITÉ & ANALYSE SCIENTIFIQUE** *texte en italique*


In [ ]:
# VII — INTERPRÉTABILITÉ & ANALYSE SCIENTIFIQUE (VERSION SÉCURISÉE)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier

# 7.1 Modèle entraîné sur l'ensemble complet
best_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
best_rf.fit(X_enriched, y_numeric)

# 7.2 Feature Importance
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

print(" TOP 10 DES VARIABLES LES PLUS IMPORTANTES (NATIVE) ")
for f in range(10):
    print(f"{f + 1}. {X_enriched.columns[indices[f]]} ({importances[indices[f]]*100:.2f}%)")

# 7.3 SHAP Values (Correction syntaxe universelle pour éviter les crashs)
print("\n=== CALCUL DES SHAP VALUES ===")
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer(X_enriched)

# Graphique SHAP global sécurisé avec la notation [..., 1]
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[..., 1], X_enriched, max_display=10, show=False)
plt.title("SHAP Summary Plot (Top 10 variables clés)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 ANALYSE DES CLUSTERS vs MALADIES RÉELLES

import pandas as pd

# Vérification simple des données
if 'X_enriched' in locals() and 'y_numeric' in locals():

    # Table de comparaison clusters vs classes réelles
    table_contingence = pd.crosstab(
        X_enriched['cluster_consensus'],
        y_numeric,
        rownames=['Clusters'],
        colnames=['Classes réelles']
    )

    print("TABLE DE CONTINGENCE (CLUSTERS vs CLASSES RÉELLES) ")
    print(table_contingence)

else:
    print("Erreur : données manquantes (X_enriched ou y_numeric).")

**Analyse de l’interprétabilité**

- Les résultats de la table de contingence montrent une séparation très claire.
Le Cluster 0 correspond presque uniquement aux feuilles saines, tandis que le Cluster 1 regroupe principalement les feuilles malades.
Cela signifie que le clustering a réussi à retrouver la structure réelle des données sans supervision.

- Les analyses d’importance des variables (Feature Importance et SHAP) confirment ce résultat.
Les variables issues du clustering (comme gmm_proba_cl1 et cluster_consensus) font partie des plus importantes du modèle, ce qui montre qu’elles apportent une information très utile pour la prédiction.

- Les embeddings comme PCA_1 jouent aussi un rôle important, ce qui montre que la réduction de dimension améliore la séparation des classes.


Les clusters ne sont pas abstraits , ils représentent bien des structures liées à l’état de santé des feuilles et améliorent clairement la performance du modèle.

**VIII- CONCLUSION**

Ce projet a permis de construire un pipeline hybride complet pour la classification des maladies des feuilles de café. En combinant apprentissage non supervisé (clustering), fusion par consensus et apprentissage supervisé, nous avons pu exploiter à la fois la structure cachée des données et les informations étiquetées.

Les résultats montrent que l’enrichissement des données par les clusters améliore globalement les performances des modèles et renforce leur stabilité. L’analyse d’interprétabilité confirme également la pertinence des variables spectrales et des features issues du clustering dans la prise de décision.

Ce travail met en évidence l’intérêt des approches hybrides pour des problèmes complexes de classification en données réelles bruitées.

**ARCHITECTURE DU PIPELINE FINAL**



DONNÉES BRUTES (data_Projet2) ➔ 0. Inspection & Préparation ➔ 1. EDA & Structure ➔ 2. Réduction de Dimension ➔ 3. Apprentissage Non Supervisé ➔ 4. Évaluation des Clusters ➔ 5. Consensus Clustering ➔ 6. Feature Engineering ➔ 7. Approche Supervisée Multi-Scénarios ➔ 8. Pseudo-Labeling Semi-Supervisé ➔ 9. Évaluation & Validation Statistique ➔ 10. Interprétabilité SHAP ➔ FIN DU PIPELINE
